# PHerc0846A Inference

Forward-only inference of any of the three mix6 models on the PHerc0846A sheets and the other mix6 scrolls. Set `CHECKPOINT` in the settings cell; the kind
of model is detected from the file (`utils/predict_models.detect_kind`) and every model reads a scroll the way it natively does:

| kind | file (where its training leaves it) | reads | prediction |
|---|---|---|---|
| `cue_dropout` | `models/runs/cue_dropout_labelsv6_mix6/best_model_loss.pth` (`train.py`) | 8 slices of a flat depth-13 surface, `DEPTH_OFFSET = -1` selects slices 9-16 | one value per 16 px cell |
| `reader_v3` | `reader-v3/runs/mix6/ckpt_002000.pth` (`finetune_readerv3.py`) | the 17 central layers of the 28-layer render | one value per pixel |
| `v8in` | `nader-fork/runs/v8in_mix6_5ep/model` (`finetune_v8in.py`) | the 24 central layers | one value per pixel |

The figures are the same for every model: prediction | composite of the layers that model reads, and for the trained scrolls a second row with the inklabel overlay
and the asf_morph cue. The inklabel is coloured by which models trained on it (`utils/holdout_exclusion.py`): red = used by all three, dark yellow = within 14 voxels of
the prize sheet (excluded from reader-v3 and cue_dropout, and from v8-in), orange = within 25.5 voxels (excluded from v8-in only). Only the prediction changes
between models. The depth ensembles, cue ablation, depth sweeps and denoising cells further down are specific to the cue_dropout model and are skipped for the other two.

Run the setup cells, then any individual segment cell. The cell near the end processes the remaining PHerc0846A quick patches into one `quick_patches` subfolder, excluding segments with
individual cells. Volumes and masks must already be assembled (`python3 assemble_training_segments.py --mix6`); this notebook does not reverse volumes, detect surfaces, or modify labels.


In [ ]:
# [] = one depth; [low, high] = ensemble every offset low..high around the usual depth ([-3, 3] -> 7 depths); cue_dropout only
depth_ensemble = []
# how ensemble depths combine: 'z-score' (top-2), 'max', 'mid-boost' (k=3), 'mean' (top-2); '' = 'mid-boost'
ensemble_method = ''

# any of the three mix6 models; the kind (cue_dropout | reader_v3 | v8in) is detected from the file:
#   /vesuvius/models/runs/cue_dropout_labelsv6_mix6/best_model_loss.pth    cue_dropout   python3 train.py
#   /vesuvius/reader-v3/runs/mix6/ckpt_002000.pth                          reader_v3     python3 finetune_readerv3.py
#   /vesuvius/nader-fork/runs/v8in_mix6_5ep/model  (or a Lightning .ckpt)  v8in          python3 finetune_v8in.py
CHECKPOINT = "reader-v3/runs/mix6/ckpt_002000.pth"
DEPTH_OFFSET = -1  # cue_dropout only: -1 selects slices 9-16 of the flat depth-13 surface
BATCH_SIZE = {"reader_v3": 4, "v8in": 16}  # inference batch of the two per-pixel models

FLAT_DEPTH = 13
EVAL_STRIDE = 32
PREVIEW_DOWNSCALE = 10  # previews shrink by this factor; saved figures stay full size
DEVICE = "auto"

import os
import sys
from pathlib import Path

REPO = Path("/vesuvius")
# nonml_ink.py (imported by utils.precomputed_asf) lives in old/
for _path in (REPO, REPO / "old"):
    if str(_path) not in sys.path:
        sys.path.insert(0, str(_path))
os.chdir(REPO)

In [ ]:
# a long-lived kernel must pick up the input_cue field and the 4-cue stem before the model is built
import importlib
import utils.mix6
import utils.config
import utils.prefilter
import utils.model
import utils.predict_models

for _module in (utils.mix6, utils.config, utils.prefilter, utils.model, utils.predict_models):
    importlib.reload(_module)

In [ ]:
import gc
import json
import tempfile
import types

import cv2
import numpy as np
import torch
import zarr
from IPython.display import display
from PIL import Image

from utils import mix6, predict_models
from utils.fast_figure import colorize, gray_rgb, save_panel_grid
from utils.platform import get_zarr_dir

Image.MAX_IMAGE_PIXELS = None
MODEL_PATH = Path(CHECKPOINT).resolve()
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Checkpoint is missing: {MODEL_PATH}")
KIND = predict_models.detect_kind(MODEL_PATH)
ZARR_DIR = Path(get_zarr_dir())
CACHE_ROOT = REPO / "_ves_tmp" / "0846A_inference"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

if KIND == "cue_dropout":
    from utils.config import Config, ScrollConfig, resolve_run_config
    from utils.model import create_model
    from utils.norm import (
        ANCHOR_GAP_LEVEL, ANCHOR_PAPYRUS_LEVEL, UNIFIED_CACHE_PATH,
        compute_norm, compute_surface_anchors, load_cached_norm,
    )
    from utils.visualizer import TensorboardVisualizer, predict_tiles

    RUN_CONFIG, RUN_CONFIG_PATH = resolve_run_config(MODEL_PATH)
    C = Config()
    for section in ("data", "model"):
        for name, value in RUN_CONFIG[section].items():
            if section == "data" and name == "scrolls":
                C.data.scrolls = [ScrollConfig(**item) for item in value]
            elif name != "probe_rois" and hasattr(getattr(C, section), name):
                setattr(getattr(C, section), name, value)
    C.model.compile_model = False
    C.tra.supcon = False
    C.tra.dann = False
    C.device = ("cuda" if torch.cuda.is_available() else "cpu") if DEVICE == "auto" else DEVICE
    C.data.eval_stride = EVAL_STRIDE
    C.data.surface_window_offset = int(DEPTH_OFFSET)
    C.data.surface_window_offset_by_scroll = {}
    C.data.zarr_path = str(ZARR_DIR)
    DEPTH = int(C.data.depth)
    if DEPTH != 8 or not C.model.surface_teacher_input or not C.data.surface_relative_depth_window:
        raise ValueError("Select a literal-surface, eight-slice checkpoint")
    if EVAL_STRIDE <= 0 or EVAL_STRIDE > C.model.multitile_grid * C.model.multitile_subtile:
        raise ValueError("Inference stride must be positive and no larger than the prediction center")
    WINDOW_START = FLAT_DEPTH - (DEPTH - 1) // 2 + int(DEPTH_OFFSET)
    WINDOW_END = WINDOW_START + DEPTH
    NORM_MODE = str(C.data.norm_mode)
    MODEL_NAME = f"{MODEL_PATH.parent.parent.name}_{MODEL_PATH.parent.name}_{MODEL_PATH.stem}"
    OUT_DIR = REPO / "output" / "0846A" / MODEL_NAME / f"offset_{DEPTH_OFFSET:+d}"
    print(f"Model: {MODEL_PATH}\nConfig: {RUN_CONFIG_PATH}\nForward slices: {WINDOW_START}-{WINDOW_END - 1} (flat depth {FLAT_DEPTH})")
else:
    C = None
    MODEL_NAME = f"{KIND}_{MODEL_PATH.parent.name}_{MODEL_PATH.stem}"
    OUT_DIR = REPO / "output" / "0846A" / MODEL_NAME
    print(f"Model: {MODEL_PATH} ({KIND}); it reads a fixed layer window of the render, nothing to configure")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# the cue_dropout model reads a flat depth-FLAT_DEPTH surface of every 0846A patch (load_inputs); detected surfaces are never used
MAIN_PATCHES = {
    "auto_grown_20260927134928530_abf": {"zid": "20260927134928", "norm_id": 20260927134931},
    "auto_grown_20261012065803387_straight_abf": {"zid": "20261012065810", "norm_id": 20261012065811},
    "auto_grown_20261012065803387_straight_abf_m28": {"zid": "20261012065812", "norm_id": 20261012065813},
    "auto_grown_20261012073244544_abf": {"zid": "20261012073244", "norm_id": 20261012073246},
    "auto_grown_20261012081237255_abf": {"zid": "20261012081237", "norm_id": 20261012081238},
    "auto_grown_20261014084609861_straight_abf": {"zid": "20261014084610", "norm_id": 20261014084611},
    "auto_grown_20261012065803387_straightened2_abf": {"zid": "20261012065814", "norm_id": 20261012065815},
    "auto_grown_20261012065803387_straightened2_abf_in_1": {"zid": "20261012065816", "norm_id": 20261012065817},
    "auto_grown_20261012065803387_straightened2_abf_out_1": {"zid": "20261012065818", "norm_id": 20261012065819},
    "auto_grown_20261012065803387_straightened2_abf_out_2": {"zid": "20261012065820", "norm_id": 20261012065821},
    "auto_grown_20261012065803387_straightened2_abf_out_3": {"zid": "20261012065822", "norm_id": 20261012065823},
    "auto_grown_20261012065803387_straightened2_abf_out_4": {"zid": "20261012065824", "norm_id": 20261012065825},
    "auto_grown_20261012065803387_straightened2_abf_out_5": {"zid": "20261012065826", "norm_id": 20261012065827},
}
# the other mix6 scrolls (trained): their stored surface labels and window offset 0, not the flat depth-13 surface of the 0846A patches
MIX6_OTHER = {
    "PHerc0009B_20250919125754": "20250919125754",
    "PHerc0343_first_letters_20261007000000": "20261007000000",
    "PHerc1447_w058_20260930144758": "20260930144758",
    "PHerc1447_w060_20260930144760": "20260930144760",
    "PHerc0841_20260221022814": "20260221022814",
    "PHerc0814_20260226000000": "20260226000000",
    "PHerc0172_w068_20251111010954": "20251111010954",
    "PHerc0172_w087_20251112000002": "20251112000002",
}
for _name, _sid in MIX6_OTHER.items():
    MAIN_PATCHES[_name] = {"zid": _sid, "norm_id": int(_sid), "surface_dir": REPO / "surface_labels" / _sid, "offset": 0}
for patch in MAIN_PATCHES.values():
    patch["mask_dir"] = REPO / "masks"
# fiber TPS sheet over straightened2 (tifxyz/fiber_tps_20261028_flatboi), rendered like a quick patch, rotated 180
FIBER_TPS = "fiber_tps_20261028_flatboi"
MAIN_PATCHES[FIBER_TPS] = {"zid": FIBER_TPS, "mask_dir": REPO / "masks" / "extras"}
WINNING = "auto_grown_20261012065803387_straightened2_abf"
# every scroll the mix6 models train on (utils/mix6.py): figures show the inklabel overlay and the asf_morph cue
TRAINING_NAMES = (
    "auto_grown_20260927134928530_abf", "auto_grown_20261012073244544_abf",
    "auto_grown_20261012065803387_straight_abf_m28",
    "auto_grown_20261012065803387_straightened2_abf_in_1", "auto_grown_20261012065803387_straightened2_abf_out_1",
    "auto_grown_20261012065803387_straightened2_abf_out_2", "auto_grown_20261012065803387_straightened2_abf_out_3",
    *MIX6_OTHER,
)
HOLDOUT_NAMES = ("auto_grown_20261012065803387_straightened2_abf_out_4",
                 "auto_grown_20261012065803387_straightened2_abf_out_5",
                 "auto_grown_20260926074031926_abf", "auto_grown_20260927143929299_abf", "auto_grown_20260927154033403_abf",
                 FIBER_TPS)
QUICK_ROOT = REPO / "tifxyz" / "quick_patches_from_D" / "PHerc0846A"
if not QUICK_ROOT.is_dir():
    raise FileNotFoundError(QUICK_ROOT)
QUICK_PATCHES = {
    path.name: {"zid": path.name, "mask_dir": REPO / "masks" / "extras"}
    for path in sorted(QUICK_ROOT.iterdir()) if path.is_dir() and (path / "x.tif").is_file()
}
INDIVIDUAL_NAMES = (WINNING, *TRAINING_NAMES, *HOLDOUT_NAMES)
PATCHES = {**QUICK_PATCHES, **MAIN_PATCHES}
REMAINING_QUICK_NAMES = tuple(name for name in QUICK_PATCHES if name not in INDIVIDUAL_NAMES)
MODEL = None


def supervision_role(name):
    patch = PATCHES[name]
    identities = {name, str(patch["zid"]), str(patch.get("norm_id", ""))}
    if KIND != "cue_dropout":  # the mix6 fine-tunes train on the mix6 corpus: the 7 0846A sheets and the 8 other scrolls
        trained = {str(alias) for alias in mix6.train_ids()} | {mix6.render_id(alias) for alias in mix6.train_ids()}
        return "TRAIN (mix6 corpus)" if identities & trained else "VIS-ONLY (not supervised by this checkpoint)"
    domains = C.data.train_scroll_dict or {}
    excluded = set(C.data.holdout_domains or [])
    trained = {str(sid) for domain, ids in domains.items() if domain not in excluded for sid in ids}
    if not domains:
        trained = {str(scroll.scroll_id) for scroll in C.data.scrolls}
    return "TRAIN (subject to saved label split)" if identities & trained else "VIS-ONLY (not supervised by this checkpoint)"


print(f"Output: {OUT_DIR}\nIndividual segments: {len(INDIVIDUAL_NAMES)}; remaining quick patches: {len(REMAINING_QUICK_NAMES)}")


In [ ]:
# the cue_dropout checkpoint's input heads must be replayed from its run config (strict loading fails on a 3-head model otherwise)
if KIND == "cue_dropout":
    if str(C.model.input_cue) != str(RUN_CONFIG["model"].get("input_cue") or ""):
        raise RuntimeError("input_cue was not replayed from the run config; restart the kernel")
    print("input heads:", 4 if C.model.input_cue else 3, C.model.input_cue or "(raw, lcn, dz)")
else:
    print(f"{KIND}: nothing to replay")

In [ ]:
def input_paths(name):
    patch = PATCHES[name]
    volume_path = ZARR_DIR / f"{patch['zid']}.zarr"
    mask_path = patch["mask_dir"] / f"{patch['zid']}.png"
    missing = [str(path) for path in (volume_path, mask_path) if not path.exists()]
    if missing:
        raise FileNotFoundError("Assemble these inputs before inference: " + ", ".join(missing))
    return volume_path, mask_path


# most recent patch's inputs (cue_dropout), so consecutive cells on one patch skip the reload
_INPUT_CACHE = {}

if KIND == "cue_dropout":
    def get_model():
        global MODEL
        if MODEL is None:
            model, parameter_count = create_model(C)
            state = torch.load(MODEL_PATH, map_location="cpu", weights_only=True)
            if "model_state_dict" in state:
                state = state["model_state_dict"]
            elif "state_dict" in state:
                state = state["state_dict"]
            state = {name.removeprefix("module.").removeprefix("_orig_mod."): value for name, value in state.items()}
            state = {name: value for name, value in state.items() if not name.startswith(("supcon_head.", "domain_head."))}
            model.load_state_dict(state, strict=True)
            model.eval()
            MODEL = model
            print(f"Loaded {parameter_count:,} parameters on {C.device}")
        return MODEL


    def flat_norm(patch, volume_path, mask_path, surface_depth, surface_confidence):
        if "norm_id" in patch and FLAT_DEPTH == 13:
            try:
                cached = load_cached_norm(patch["norm_id"], UNIFIED_CACHE_PATH, mode=NORM_MODE)
            except RuntimeError:
                cached = None
            if cached is not None:
                return cached
        if NORM_MODE == "raw255":
            return 0.0, 1.0, 0.0, 255.0
        private_cache = str(CACHE_ROOT / "global_norm.json")
        if NORM_MODE == "global":
            cached = load_cached_norm(patch["zid"], UNIFIED_CACHE_PATH, mode="global")
            cached = cached or load_cached_norm(patch["zid"], private_cache, mode="global")
            return cached or compute_norm(patch["zid"], str(ZARR_DIR), private_cache, mask_dir=str(patch["mask_dir"]))
        if NORM_MODE not in ("surface_anchor", "surface_anchor_knee"):
            raise ValueError(f"Unsupported normalization: {NORM_MODE}")
        attrs_path = volume_path / ".zattrs"
        signature = {
            "volume": str(volume_path.resolve()), "flat_depth": FLAT_DEPTH,
            "attrs_mtime": attrs_path.stat().st_mtime_ns if attrs_path.exists() else volume_path.stat().st_mtime_ns,
            "mask_mtime": mask_path.stat().st_mtime_ns, "shape": list(surface_depth.shape),
        }
        cache_path = CACHE_ROOT / f"{patch['zid']}_flat_anchors.json"
        if cache_path.is_file():
            cached = json.loads(cache_path.read_text())
            if cached.get("signature") == signature:
                return tuple(cached["norm"])
        with tempfile.TemporaryDirectory(dir=CACHE_ROOT) as temporary:
            surface_dir = Path(temporary) / str(patch["zid"])
            surface_dir.mkdir()
            np.save(surface_dir / "depth.npy", surface_depth)
            np.save(surface_dir / "confidence.npy", surface_confidence)
            anchors = compute_surface_anchors(patch["zid"], str(ZARR_DIR), surface_dir=temporary,
                                              mask_dir=str(patch["mask_dir"]))
        span = (anchors["papyrus"] - anchors["gap"]) / (ANCHOR_PAPYRUS_LEVEL - ANCHOR_GAP_LEVEL)
        lower = anchors["gap"] - ANCHOR_GAP_LEVEL * span
        norm = (0.0, 1.0, lower, lower + span)
        cache_path.write_text(json.dumps({"signature": signature, "norm": norm, "anchors": anchors}, indent=2))
        return norm


    def flat_cue(patch, volume, mask):
        """precomputed cue map for a model with input_cue 'pre_clahe': the stored map of the patch's flat alias when there is
        one (the exact map training read), else computed from the loaded window and cached under CACHE_ROOT/cue."""
        from utils.precomputed_asf import cue_from_window

        stored = Path(C.data.cue_dir) / f"{patch.get('norm_id', patch['zid'])}.npy"
        if stored.is_file():
            return np.load(stored, mmap_mode="r")
        cached = CACHE_ROOT / "cue" / f"{patch['zid']}_s{WINDOW_START}.npy"
        if not cached.is_file():
            cached.parent.mkdir(parents=True, exist_ok=True)
            cue = cue_from_window(np.asarray(volume[WINDOW_START:WINDOW_END], dtype=np.float32), mask)
            np.save(cached, cue)
        return np.load(cached, mmap_mode="r")


    def load_inputs(name):
        import time

        volume_path, mask_path = input_paths(name)
        key = (name, (volume_path / ".zarray").stat().st_mtime_ns, mask_path.stat().st_mtime_ns)
        if key in _INPUT_CACHE:
            return _INPUT_CACHE[key]
        _INPUT_CACHE.clear()
        gc.collect()
        source = zarr.open(str(volume_path), mode="r")
        if source.attrs.get("layers_reversed_from") or source.attrs.get("facing_reversed"):
            raise ValueError(f"Forward inference requires the original volume: {volume_path}")
        with Image.open(mask_path) as image:
            mask = np.asarray(image.convert("L")) > 0
        height, width = mask.shape
        if tuple(source.shape[1:]) != (height, width):
            raise ValueError(f"Mask and volume frames differ for {name}: {mask.shape} vs {source.shape}")
        if not mask.any():
            raise ValueError(f"Invalid mask for {name}")
        started = time.time()
        # small zarr chunks make the overlapping context-strip reads decompress each chunk many times
        volume = np.ascontiguousarray(source[:])
        volume.setflags(write=False)
        print(f"[preload] {name}: {volume.shape} {volume.nbytes / 1024**3:.2f} GiB in RAM in {time.time() - started:.1f}s")
        patch = PATCHES[name]
        offset = int(patch.get("offset", DEPTH_OFFSET))
        if patch.get("surface_dir"):
            surface_depth = np.load(Path(patch["surface_dir"]) / "depth.npy")
            surface_confidence = np.load(Path(patch["surface_dir"]) / "confidence.npy")
            if surface_depth.shape != mask.shape:
                raise ValueError(f"Surface labels {surface_depth.shape} differ from the mask {mask.shape} for {name}")
        else:
            surface_depth = np.where(mask, FLAT_DEPTH, 255).astype(np.uint8)
            surface_confidence = np.where(mask, 255, 0).astype(np.uint8)
        valid_depth = surface_depth[(surface_depth != 255) & mask]
        win_start = int(np.median(valid_depth)) - (DEPTH - 1) // 2 + offset
        win_end = win_start + DEPTH
        coords = TensorboardVisualizer._gen_tile_coords(
            types.SimpleNamespace(c=C), (C.data.d_start, C.data.d_end),
            (0, height), (0, width), mask, z_step=DEPTH,
        )
        if not coords or any(coordinate[0] != 0 for coordinate in coords):
            raise ValueError(f"Expected one nonempty surface-relative pass for {name}")
        if win_start < 0 or win_end > source.shape[0]:
            raise ValueError(f"Invalid depth window {win_start}:{win_end} for {name}")
        norm = flat_norm(PATCHES[name], volume_path, mask_path, surface_depth, surface_confidence)
        cue = flat_cue(PATCHES[name], volume, mask) if str(C.model.input_cue) == "pre_clahe" else None
        _INPUT_CACHE[key] = {"volume": volume, "mask": mask, "surface_depth": surface_depth,
                             "surface_confidence": surface_confidence, "coords": coords, "norm": norm, "cue": cue,
                             "offset": offset, "win_start": win_start, "win_end": win_end}
        return _INPUT_CACHE[key]

In [ ]:
import importlib
import utils.fast_figure as figure_utils
import utils.depth_ensemble as depth_ensemble_utils
import utils.holdout_exclusion as holdout_exclusion

importlib.reload(figure_utils)
importlib.reload(depth_ensemble_utils)
importlib.reload(holdout_exclusion)
depth_ensemble_utils.resolve(depth_ensemble, ensemble_method)

PREDICTIONS = {}  # last prediction of each rendered patch, for cells that post-process it

# inklabel colours by which mix6 models trained on the pixel (prize-sheet exclusion masks of the 0846A sheets)
# red = used by all three, dark yellow = within 14 vox (no reader-v3 / cue_dropout / v8-in), orange = 14-25.5 vox (no v8-in)
LABEL_ALL, LABEL_NOT_READER_CUE, LABEL_NOT_V8IN = (220, 0, 0), (190, 160, 0), (255, 140, 0)


def native_inputs(name):
    """reader_v3 / v8in: papyrus mask and the max composite of the layers the model reads (no volume is held in memory)."""
    volume_path, mask_path = input_paths(name)
    source = zarr.open(str(volume_path), mode="r")
    if source.attrs.get("layers_reversed_from") or source.attrs.get("facing_reversed"):
        raise ValueError(f"Forward inference requires the original volume: {volume_path}")
    with Image.open(mask_path) as image:
        mask = np.asarray(image.convert("L")) > 0
    if tuple(source.shape[1:]) != mask.shape:
        raise ValueError(f"Mask and volume frames differ for {name}: {mask.shape} vs {source.shape}")
    start, end = predict_models.native_window(KIND, source.shape[0], MODEL_PATH)
    return volume_path, mask, start, end


def predict_native(name):
    """the per-pixel prediction (reader_v3 | v8in) of a patch, cached under CACHE_ROOT/MODEL_NAME; NaN outside the papyrus."""
    volume_path, mask, start, end = native_inputs(name)
    cache = CACHE_ROOT / MODEL_NAME / name
    if KIND == "reader_v3":
        prediction = predict_models.predict_reader(volume_path, MODEL_PATH, cache / "reader_v3.tif", BATCH_SIZE["reader_v3"])
    else:
        prediction = predict_models.predict_v8in(volume_path, MODEL_PATH, cache, BATCH_SIZE["v8in"])
    framed = np.full(mask.shape, np.nan, np.float32)
    rows, cols = min(mask.shape[0], prediction.shape[0]), min(mask.shape[1], prediction.shape[1])
    framed[:rows, :cols] = prediction[:rows, :cols]
    framed[~mask] = np.nan
    projection = np.asarray(zarr.open(str(volume_path), mode="r")[start:end]).max(axis=0)
    return framed, mask, np.clip(projection, 0, 255).astype(np.uint8) * mask, start, end


def training_label_and_cue(name, mask):
    """inklabel (dilated_inklabels) and asf_morph cue (cue_asf_morph) of a training patch, both in the mask frame."""
    patch = PATCHES[name]
    with Image.open(REPO / "dilated_inklabels" / f"{patch['norm_id']}.png") as image:
        label = np.asarray(image.convert("L")) > 0
    cue = np.load(REPO / "cue_asf_morph" / f"{patch['norm_id']}.npy")
    if label.shape != mask.shape or cue.shape != mask.shape:
        raise ValueError(f"{name}: mask {mask.shape}, label {label.shape}, cue {cue.shape} differ")
    return label, cue


def label_layers(name, label):
    """(mask, colour) layers of the inklabel: kept by every model, dropped by v8-in only, dropped by all three."""
    alias = PATCHES[name]["norm_id"]
    near = holdout_exclusion.load(alias, label.shape)
    far = holdout_exclusion.load(alias, label.shape, directory=holdout_exclusion.V8IN_EXCLUSION_DIR)
    near = np.zeros_like(label) if near is None else near
    far = near if far is None else far | near
    return [(label & ~far, LABEL_ALL), (label & far & ~near, LABEL_NOT_V8IN), (label & near, LABEL_NOT_READER_CUE)]


def render(name, output_subdir=None, preview=True):
    from utils.fast_figure import color_overlay, orient_display

    print(f"{name}: {supervision_role(name)}")
    if KIND == "cue_dropout":
        inputs = load_inputs(name)
        model = get_model()
        height, width = inputs["mask"].shape
        offset, WINDOW_START, WINDOW_END = inputs["offset"], inputs["win_start"], inputs["win_end"]
        offsets, method = depth_ensemble_utils.resolve(depth_ensemble, ensemble_method)
        prediction = depth_ensemble_utils.ensemble_predict(predict_tiles, offsets, method)(
            C, model, inputs["volume"], inputs["mask"], inputs["coords"],
            (0, height), (0, width), C.data.d_start, name, *inputs["norm"],
            surface_depth_map=inputs["surface_depth"],
            surface_confidence_map=inputs["surface_confidence"],
            surface_depth_offset=offset, cue_map=inputs["cue"],
        )
        prediction = np.array(prediction, dtype=np.float32)  # one value per 16 px cell
        mask = inputs["mask"]
        projection = np.asarray(inputs["volume"][WINDOW_START:WINDOW_END]).max(axis=0)
        composite = np.clip(projection, 0, 255).astype(np.uint8) * mask
        ensemble_label = depth_ensemble_utils.describe([offset + o for o in offsets], method)
        pred_interpolation = cv2.INTER_NEAREST
        del inputs, projection
    else:
        prediction, mask, composite, WINDOW_START, WINDOW_END = predict_native(name)  # one value per pixel
        ensemble_label = ""
        pred_interpolation = cv2.INTER_AREA
    PREDICTIONS[name] = prediction
    display_prediction = orient_display(prediction, name)
    display_composite = orient_display(composite, name)
    pred_title = f"{name} | {ensemble_label or f'forward {WINDOW_START}-{WINDOW_END - 1}'}"
    pred_rgb = colorize(display_prediction, "gray_r", (0.78, 0.86, 0.95))
    composite_panel = (f"composite {WINDOW_START}-{WINDOW_END - 1}", gray_rgb(display_composite), cv2.INTER_AREA, 9.362)
    if name in TRAINING_NAMES:
        # 2x2: prediction | composite over prediction + inklabel coloured by which models trained on it | asf_morph cue
        label, cue = training_label_and_cue(name, mask)
        cell_hw = display_composite.shape
        pred_full = cv2.resize(pred_rgb, (cell_hw[1], cell_hw[0]), interpolation=cv2.INTER_NEAREST)
        overlay = color_overlay(pred_full, [(orient_display(layer, name), color) for layer, color in label_layers(name, label)])
        panels = [
            [(pred_title, pred_full, cv2.INTER_AREA, None), composite_panel],
            [(f"{name} | inklabel overlay", overlay, cv2.INTER_AREA, None),
             ("asf_morph cue", gray_rgb(orient_display(cue, name)), cv2.INTER_AREA, None)],
        ]
        del pred_full, overlay, label, cue
    else:
        panels = [[(pred_title, pred_rgb, pred_interpolation, None), composite_panel]]
    folder = OUT_DIR if output_subdir is None else OUT_DIR / output_subdir
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / f"{name}.png"
    save_panel_grid(panels, display_composite.shape, str(output_path), gap_px=24)
    if preview:
        with Image.open(output_path) as saved:
            reduced = saved.resize((max(1, saved.width // PREVIEW_DOWNSCALE),
                                    max(1, saved.height // PREVIEW_DOWNSCALE)), Image.Resampling.LANCZOS)
        display(reduced)
    print(f"Saved {output_path}")
    del prediction, composite, display_prediction, display_composite, panels
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return output_path


def render_remaining_quick_patches():
    for name in REMAINING_QUICK_NAMES:
        input_paths(name)
    failures = {}
    outputs = []
    for index, name in enumerate(REMAINING_QUICK_NAMES, start=1):
        print(f"[{index}/{len(REMAINING_QUICK_NAMES)}] {name}")
        try:
            outputs.append(render(name, output_subdir="quick_patches", preview=False))
        except Exception as error:
            failures[name] = str(error)
            print(f"FAILED {name}: {error}")
    print(f"Saved {len(outputs)} quick-patch figures to {OUT_DIR / 'quick_patches'}")
    if failures:
        raise RuntimeError(f"Quick-patch failures: {failures}")
    return outputs


In [ ]:
# HELD OUT WINNING SHEET (straightened2)
render(WINNING)

In [ ]:
# TRAIN: ABF
render("auto_grown_20260927134928530_abf")

In [ ]:
# TRAIN: 073246
render("auto_grown_20261012073244544_abf")


In [ ]:
# TRAIN: m28 (straight sheet at offset +29, -29 in VC3D)
render("auto_grown_20261012065803387_straight_abf_m28")


In [ ]:
# TRAIN: straightened2 in_1
render("auto_grown_20261012065803387_straightened2_abf_in_1")


In [ ]:
# TRAIN: straightened2 out_1
render("auto_grown_20261012065803387_straightened2_abf_out_1")


In [ ]:
# TRAIN: straightened2 out_2
render("auto_grown_20261012065803387_straightened2_abf_out_2")


In [ ]:
# TRAIN: straightened2 out_3
render("auto_grown_20261012065803387_straightened2_abf_out_3")


In [ ]:
# TRAIN: PHerc0009B (its stored surface labels, window offset 0, stored cue)
render("PHerc0009B_20250919125754")

In [ ]:
# TRAIN: PHerc0343 first letters (its stored surface labels, window offset 0, stored cue)
render("PHerc0343_first_letters_20261007000000")

In [ ]:
# TRAIN: PHerc1447 w058 (its stored surface labels, window offset 0, stored cue)
render("PHerc1447_w058_20260930144758")

In [ ]:
# TRAIN: PHerc1447 w060 (its stored surface labels, window offset 0, stored cue)
render("PHerc1447_w060_20260930144760")

In [ ]:
# TRAIN: PHerc0841 (its stored surface labels, window offset 0, stored cue)
render("PHerc0841_20260221022814")

In [ ]:
# TRAIN: PHerc0814 (its stored surface labels, window offset 0, stored cue)
render("PHerc0814_20260226000000")

In [ ]:
# TRAIN: PHerc0172 w068 (its stored surface labels, window offset 0, stored cue)
render("PHerc0172_w068_20251111010954")

In [ ]:
# TRAIN: PHerc0172 w087 (its stored surface labels, window offset 0, stored cue)
render("PHerc0172_w087_20251112000002")

In [ ]:
# HELD OUT: straightened2 out_4
render("auto_grown_20261012065803387_straightened2_abf_out_4")


In [ ]:
# HELD OUT: straightened2 out_5
render("auto_grown_20261012065803387_straightened2_abf_out_5")

In [ ]:
# HELD OUT: quick patch 20260926074031926
render("auto_grown_20260926074031926_abf")


In [ ]:
# HELD OUT: quick patch 20260927143929299
render("auto_grown_20260927143929299_abf")


In [ ]:
# HELD OUT: quick patch 20260927154033403
render("auto_grown_20260927154033403_abf")


In [ ]:
# HELD OUT: fiber TPS sheet over straightened2 (rotated 180)
render(FIBER_TPS)


In [ ]:
# VIS-ONLY: remaining quick patches; individual entries above are excluded.
quick_patch_outputs = render_remaining_quick_patches()